# Contradictory, My Dear Watson — Kaggle submission

Inference-only notebook. Loads the already-trained `xlm-roberta-large-xnli`
checkpoint (92.99% val_accuracy — see this project's `README.md`/`CLAUDE.md`
for how it was trained) and predicts on the competition's `test.csv`.

**Before running ("Save & Run All"), do these two things in this Kaggle
notebook's settings:**

1. **Add Input → Datasets** → attach a private dataset containing the file
   `nli_large_best.pt` (create it beforehand: Kaggle → Datasets → New
   Dataset → upload `checkpoints/nli_large_best.pt` from this repo, ~2.24GB).
   Update `CHECKPOINT_PATH` below to match the dataset slug Kaggle assigns.
2. **Settings → Internet → On** — needed once, to download the tokenizer
   and base model architecture for `joeddav/xlm-roberta-large-xnli` from
   Hugging Face Hub. The fine-tuned weights themselves come from the
   attached checkpoint, not from the Hub.

Output: `submission.csv`, picked up automatically by Kaggle's
"Submit to Competition" after a successful run.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

TEST_CSV_PATH = '/kaggle/input/contradictory-my-dear-watson/test.csv'
CHECKPOINT_PATH = '/kaggle/input/watson-nli-large-checkpoint/nli_large_best.pt'
MODEL_NAME = 'joeddav/xlm-roberta-large-xnli'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
max_len = 128  # covers 99.4% of premise+hypothesis pairs without truncation

def bert_encode(premises, hypotheses, tokenizer, max_len=max_len):
    encodings = tokenizer(
        list(premises), list(hypotheses),
        padding='max_length', truncation=True, max_length=max_len,
        return_tensors='pt')
    input_ids = encodings['input_ids']
    if 'token_type_ids' in encodings:
        token_type_ids = encodings['token_type_ids']
    else:
        token_type_ids = torch.zeros_like(input_ids)
    return {
        'input_ids': input_ids,
        'attention_mask': encodings['attention_mask'],
        'token_type_ids': token_type_ids,
    }

In [ ]:
class NLIModel(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.0, pooling='cls'):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                             token_type_ids=token_type_ids)
        if self.pooling == 'mean':
            mask = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
            counts = torch.clamp(mask.sum(dim=1), min=1e-9)
            pooled = summed / counts
        else:
            pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = NLIModel(MODEL_NAME, dropout=0.4, pooling='mean').float().to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

In [ ]:
test = pd.read_csv(TEST_CSV_PATH)
test_input = bert_encode(test.premise.values, test.hypothesis.values, tokenizer)

batch_size = 32
predictions = []
with torch.no_grad():
    for i in range(0, len(test), batch_size):
        input_ids = test_input['input_ids'][i:i+batch_size].to(device)
        attention_mask = test_input['attention_mask'][i:i+batch_size].to(device)
        token_type_ids = test_input['token_type_ids'][i:i+batch_size].to(device)
        logits = model(input_ids, attention_mask, token_type_ids)
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())

submission = test.id.copy().to_frame()
submission['prediction'] = predictions
submission.to_csv('submission.csv', index=False)
print(f'submission.csv written: {len(submission)} rows')
submission.head()